<a href="https://colab.research.google.com/github/Aymanelok/TP_AI/blob/main/TP1_Aymane_AITOUHAMMOU_Preprocessing_Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP 1 — Prétraitement des Données et Construction de Pipelines avec Scikit-learn

Aymane AITOUHAMMOU


---

## Objectifs pédagogiques

À l'issue de ce travail pratique, l'étudiant sera capable de :

1. **Configurer** un environnement de travail Python adapté au machine learning.
2. **Appliquer** les techniques fondamentales de prétraitement : encodage des variables catégorielles et normalisation.
3. **Maîtriser** le principe de non-contamination entre les ensembles d'entraînement et de test (*data leakage*).
4. **Construire** des pipelines scikit-learn de complexité croissante pour automatiser et reproductibiliser les chaînes de traitement.

---

## Contexte

En machine learning, la phase de **prétraitement** des données est déterminante pour la qualité des modèles appris. Elle comprend, entre autres, la gestion des variables catégorielles (encodage), la mise à l'échelle des variables numériques (normalisation ou standardisation), ainsi que la sélection de caractéristiques pertinentes. L'objet `Pipeline` de scikit-learn permet d'enchaîner ces transformations de manière structurée, garantissant la cohérence du traitement entre les phases d'entraînement et d'inférence.

---

## Section 0 — Importation des Bibliothèques

Nous chargeons l'ensemble des bibliothèques nécessaires à l'exécution de ce TP.

| Bibliothèque | Rôle |
|---|---|
| `numpy` | Calcul numérique matriciel |
| `pandas` | Manipulation et exploration de données tabulaires |
| `seaborn` | Accès aux jeux de données de démonstration et visualisation |
| `sklearn` | Outils de machine learning : prétraitement, sélection, modélisation |

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import warnings

from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    OrdinalEncoder,
    OneHotEncoder,
    PolynomialFeatures,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.datasets import make_regression

warnings.filterwarnings("ignore", category=FutureWarning)

print("✓ Bibliothèques chargées avec succès.")

✓ Bibliothèques chargées avec succès.


---

## Section 1 — Chargement et Exploration du Jeu de Données

Nous utilisons le jeu de données **Diamonds** disponible via la bibliothèque Seaborn. Ce dataset recense les caractéristiques de **53 940 diamants** et constitue un exemple canonique pour illustrer les problématiques de prétraitement mixte (variables numériques et catégorielles ordonnées).

### Variables du dataset

| Variable | Type | Description |
|---|---|---|
| `carat` | Numérique | Poids du diamant |
| `cut` | Catégorielle ordinale | Qualité de la taille |
| `color` | Catégorielle ordinale | Couleur (D → J) |
| `clarity` | Catégorielle ordinale | Clarté |
| `depth`, `table`, `x`, `y`, `z` | Numérique | Dimensions physiques |
| `price` | Numérique | Prix en USD (variable cible) |

In [ ]:
# Chargement du dataset
df = sns.load_dataset("diamonds")

print(f"Dimensions du dataset : {df.shape[0]} observations × {df.shape[1]} variables")
print(f"\nTypes de variables :\n{df.dtypes}")
print(f"\nValeurs manquantes :\n{df.isnull().sum()}")

Dimensions du dataset : 53940 observations × 10 variables

Types de variables :
carat       float64
cut        category
color      category
clarity    category
depth       float64
table       float64
price         int64
x           float64
y           float64
z           float64
dtype: object

Valeurs manquantes :
carat      0
cut        0
color      0
clarity    0
depth      0
table      0
price      0
x          0
y          0
z          0
dtype: int64


In [ ]:
# Affichage des premières observations
df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


---

## Section 2 — Prétraitement Manuel Étape par Étape

### 2.1 Partitionnement Train / Test

> **Principe fondamental — Isolation du jeu de test :**  
> Avant toute transformation, il est impératif de séparer les données en deux ensembles disjoints :
> - **Ensemble d'entraînement** (*train set*) : utilisé pour **ajuster** les transformateurs (`.fit_transform()`).
> - **Ensemble de test** (*test set*) : utilisé uniquement pour **évaluer** les transformations apprises (`.transform()`), sans jamais influencer leur estimation.
>
> Appliquer `.fit()` sur l'ensemble de test constitue une fuite de données (*data leakage*) et biaise les métriques de performance.

In [ ]:
train_set, test_set = train_test_split(df, test_size=0.2, random_state=0)

print(f"Taille de l'ensemble d'entraînement : {train_set.shape}")
print(f"Taille de l'ensemble de test         : {test_set.shape}")

Taille de l'ensemble d'entraînement : (43152, 10)
Taille de l'ensemble de test         : (10788, 10)


### 2.2 Encodage des Variables Catégorielles

Le jeu de données Diamonds contient trois variables catégorielles : `cut`, `color` et `clarity`. Le choix de la méthode d'encodage dépend de la nature ordinale ou nominale de chaque variable.

#### Règle de décision

| Situation | Méthode recommandée | Justification |
|---|---|---|
| Les catégories suivent un **ordre naturel** | `OrdinalEncoder` | Préserve l'information de rang |
| Les catégories sont **sans ordre** (nominales) | `OneHotEncoder` | Évite d'imposer un ordre artificiel |

> **Exercice préliminaire :** Effectuez une recherche sur la classification gemmologique des diamants afin de déterminer si les variables `cut`, `color` et `clarity` sont ordinales ou nominales.

In [ ]:
# Exploration des modalités de chaque variable catégorielle
for col in ['cut', 'color', 'clarity']:
    print(f"Modalités de '{col}' : {df[col].unique().tolist()}")

Modalités de 'cut' : ['Ideal', 'Premium', 'Good', 'Very Good', 'Fair']
Modalités de 'color' : ['E', 'I', 'J', 'H', 'F', 'G', 'D']
Modalités de 'clarity' : ['SI2', 'SI1', 'VS1', 'VS2', 'VVS2', 'VVS1', 'I1', 'IF']


In [ ]:
# Définition des ordres gemmologiques
# Source : classification GIA (Gemological Institute of America)
cut_order     = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']          # ordre croissant de qualité
color_order   = ['J', 'I', 'H', 'G', 'F', 'E', 'D']                       # D = plus incolore (meilleur)
clarity_order = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']  # IF = Internally Flawless

# Instanciation et ajustement de l'encodeur ordinal sur le train set
encoder = OrdinalEncoder(categories=[cut_order, color_order, clarity_order])
encoding_train = encoder.fit_transform(train_set[['cut', 'color', 'clarity']])

# Injection des résultats encodés dans le train set
# Remarque : on travaille sur une copie pour éviter les SettingWithCopyWarning
train_set = train_set.copy()
train_set[['cut', 'color', 'clarity']] = encoding_train

print("Encodage de l'ensemble d'entraînement — aperçu :")
train_set.head()

Encodage de l'ensemble d'entraînement — aperçu :


,carat,cut,color,clarity,depth,table,price,x,y,z
26250,1.63,4.0,3.0,4.0,61.7,55.0,15697,7.56,7.60,4.68
31510,0.34,4.0,3.0,3.0,62.2,57.0,765,4.47,4.44,2.77
40698,0.40,4.0,5.0,5.0,61.7,56.0,1158,4.73,4.77,2.93
42634,0.58,3.0,2.0,2.0,62.1,55.0,1332,5.38,5.35,3.33
47714,0.63,2.0,6.0,2.0,62.8,57.0,1885,5.40,5.46,3.41


### 2.3 Normalisation des Variables Numériques

La normalisation **Min-Max** (*Min-Max Scaling*) projette chaque variable dans l'intervalle $[0, 1]$ selon la formule :

$$x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

> **Important :** Le `MinMaxScaler` est **ajusté sur le train set** uniquement (les valeurs $x_{\min}$ et $x_{\max}$ sont estimées sur le train set). Ces mêmes paramètres sont ensuite appliqués au test set via `.transform()`, sans recalcul.

In [ ]:
scaler = MinMaxScaler()
train_set_scaled = scaler.fit_transform(train_set)

print(f"Dimensions après normalisation : {train_set_scaled.shape}")
print(f"Vérification — min : {train_set_scaled.min():.2f}, max : {train_set_scaled.max():.2f}")

Dimensions après normalisation : (43152, 10)
Vérification — min : 0.00, max : 1.00


### 2.4 Application au Jeu de Test

Les transformateurs (encodeur + scaler) **déjà ajustés** sur le train set sont appliqués au test set via `.transform()` exclusivement.

In [ ]:
# Application de l'encodage au test set (sans ré-ajustement)
test_set = test_set.copy()
encoding_test = encoder.transform(test_set[['cut', 'color', 'clarity']])
test_set[['cut', 'color', 'clarity']] = encoding_test

# Application de la normalisation au test set
test_set_scaled = scaler.transform(test_set)

print(f"Test set — dimensions : {test_set_scaled.shape}")
print(f"Test set — min : {test_set_scaled.min():.4f}, max : {test_set_scaled.max():.4f}")
# Remarque : les valeurs peuvent légèrement dépasser [0,1] si le test set contient
# des valeurs hors de la plage du train set, ce qui est attendu.

Test set — dimensions : (10788, 10)
Test set — min : -0.0001, max : 1.4444


---

## Section 3 — Automatisation avec les Pipelines Scikit-learn

### 3.1 Motivation

L'approche manuelle de la section précédente, bien qu'instructive, est sujette à des erreurs et difficile à maintenir. L'objet `Pipeline` de scikit-learn résout ces problèmes en enchaînant les transformations de façon déclarative. Ses avantages principaux sont :

- **Reproductibilité** : la chaîne de traitement est entièrement décrite dans un seul objet.
- **Sécurité** : `.fit_transform()` sur le train et `.transform()` sur le test sont gérés automatiquement.
- **Compatibilité** : une `Pipeline` s'intègre nativement dans `GridSearchCV` pour la recherche d'hyperparamètres.

### 3.2 Pipeline Simple (Encodage + Normalisation)

In [ ]:
# Rechargement du dataset original (état brut, avant toute transformation)
df = sns.load_dataset("diamonds")
train_set, test_set = train_test_split(df, test_size=0.2, random_state=0)

# Identification des colonnes
categorical_cols = ['cut', 'color', 'clarity']
numeric_cols = [c for c in df.columns if c not in categorical_cols]

# Construction de la pipeline avec ColumnTransformer
column_transformer = ColumnTransformer(
    transformers=[
        ("Encodeur",
         OrdinalEncoder(categories=[
             ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal'],
             ['J', 'I', 'H', 'G', 'F', 'E', 'D'],
             ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']
         ]),
         categorical_cols)
    ],
    remainder="passthrough"
)

pipeline_simple = Pipeline(steps=[
    ("Preprocessing", column_transformer),
    ("Normaliseur",   MinMaxScaler())
])

# Ajustement et transformation du train set
train_transformed = pipeline_simple.fit_transform(train_set)

# Transformation du test set
test_transformed  = pipeline_simple.transform(test_set)

print(f"Train set transformé — shape : {train_transformed.shape}")
print(f"Test set transformé  — shape : {test_transformed.shape}")

pipeline_simple


Train set transformé — shape : (43152, 10)
Test set transformé  — shape : (10788, 10)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('Preprocessing', ...), ('Normaliseur', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Encodeur', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

### 3.3 Pipeline Composée avec `ColumnTransformer`

Lorsque les variables numériques et catégorielles requièrent des traitements différents, on utilise un `ColumnTransformer` pour appliquer des transformations **en parallèle** sur des sous-ensembles de colonnes, avant de les réunir dans une pipeline globale.

```
Données brutes
     │
     ├── Variables catégorielles ──> OrdinalEncoder ──┐
     │                                                 ├──> MinMaxScaler ──> Données prêtes
     └── Variables numériques ──────> (passthrough) ──┘
```

In [ ]:
# Définition des ordres catégoriels
cut_order     = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
color_order   = ['J', 'I', 'H', 'G', 'F', 'E', 'D']
clarity_order = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

categorical_cols = ['cut', 'color', 'clarity']

# ColumnTransformer : encodage des colonnes catégorielles, le reste passe sans modification
column_transformer = ColumnTransformer(
    transformers=[
        ("Encodeur",
         OrdinalEncoder(categories=[cut_order, color_order, clarity_order]),
         categorical_cols)
    ],
    remainder="passthrough"
)

# Pipeline globale
pipeline_composee = Pipeline(steps=[
    ("Preprocessing", column_transformer),
    ("Normalisation", MinMaxScaler())
])

# Entraînement
train_out = pipeline_composee.fit_transform(train_set)
test_out  = pipeline_composee.transform(test_set)

print(f"Sortie train set — shape : {train_out.shape}")
print(f"Sortie test set  — shape : {test_out.shape}")

pipeline_composee

Sortie train set — shape : (43152, 10)
Sortie test set  — shape : (10788, 10)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('Preprocessing', ...), ('Normalisation', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Encodeur', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

---

## Section 4 — Exercices : Construction de Pipelines de Complexité Croissante

Les exercices suivants vous guident dans la construction de quatre pipelines de niveau croissant, depuis une pipeline de feature engineering simple jusqu'à une pipeline de bout en bout intégrant un modèle de régression.

---

### Exercice 1 — Pipeline Niveau 1 : Feature Engineering + Sélection + Normalisation

**Objectif :** Sur un jeu de données de régression synthétique, générer des caractéristiques polynomiales, sélectionner les `k` meilleures, puis normaliser.

```
Données numériques
     │
     ├──> PolynomialFeatures(degree=2)
     ├──> SelectKBest(f_regression, k=5)
     └──> MinMaxScaler
```

In [ ]:
# Génération d'un jeu de données de régression synthétique
X_reg, y_reg = make_regression(
    n_samples=100,
    n_features=4,
    noise=0.1,
    random_state=42
)
print(f"Dataset de régression — X : {X_reg.shape}, y : {y_reg.shape}")

# Construction de la pipeline Niveau 1
pipe_niveau1 = Pipeline(steps=[
    ("FeatureEngineering",  PolynomialFeatures(degree=2, include_bias=True)),
    ("SelectionFeatures",   SelectKBest(score_func=f_regression, k=5)),
    ("Normalisation",       MinMaxScaler())
])

# Application
X_niveau1 = pipe_niveau1.fit_transform(X_reg, y_reg)

print(f"\nNombre de features avant pipeline : {X_reg.shape[1]}")
print(f"Nombre de features après pipeline  : {X_niveau1.shape[1]}")
print(f"(PolynomialFeatures degree=2 sur 4 variables → {PolynomialFeatures(degree=2).fit(X_reg).n_output_features_} features, puis SelectKBest garde 5)")

pipe_niveau1

Dataset de régression — X : (100, 4), y : (100,)

Nombre de features avant pipeline : 4
Nombre de features après pipeline  : 5
(PolynomialFeatures degree=2 sur 4 variables → 15 features, puis SelectKBest garde 5)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('FeatureEngineering', ...), ('SelectionFeatures', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"degree degree: int or tuple (min_degree, max_degree), default=2If a single int is given, it specifies the maximal degree of thepolynomial features. If a tuple `(min_degree, max_degree)` is passed,then `min_degree` is the minimum and `max_degree` is the maximumpolynomial degree of the generated features. Note that `min_degree=0`and `min_degree=1` are equivalent as outputting the degree zero term isdetermined by `include_bias`.",2
,"interaction_only interaction_only: bool, default=FalseIf `True`, only interaction features are produced: features that areproducts of at most `degree` *distinct* input features, i.e. terms withpower of 2 or higher of the same input feature are excluded:- included: `x[0]`, `x[1]`, `x[0] * x[1]`, etc.- excluded: `x[0] ** 2`, `x[0] ** 2 * x[1]`, etc.",False
,"include_bias include_bias: bool, default=TrueIf `True` (default), then include a bias column, the feature in whichall polynomial powers are zero (i.e. a column of ones - acts as anintercept term in a linear model).",True
,"order order: {'C', 'F'}, default='C'Order of output array in the dense case. `'F'` order is faster tocompute, but may slow down subsequent estimators... versionadded:: 0.21",'C'
,"score_func score_func: callable, default=f_classifFunction taking two arrays X and y, and returning a pair of arrays(scores, pvalues) or a single array with scores.Default is f_classif (see below ""See Also""). The default function onlyworks with classification tasks... versionadded:: 0.18",<function f_r...001757FAAD940>
,"k k: int or ""all"", default=10Number of top features to select.The ""all"" option bypasses selection, for use in a parameter search.",5
,"feature_range feature_range: tuple (min, max), default=(0, 1)Desired range of transformed data.","(0, ...)"


---

### Exercice 2 — Pipeline Niveau 2 : Prétraitement Mixte (Numérique + Catégoriel)

**Objectif :** Sur un jeu de données mixte (variables numériques et catégorielles nominales), appliquer des transformations différenciées via un `ColumnTransformer`.

```
Données mixtes
     ├── Variables numériques  ──> MinMaxScaler
     └── Variables catégorielles ──> OneHotEncoder
```

In [ ]:
# Génération d'un dataset mixte synthétique
np.random.seed(42)
n_samples = 100

X_numeric       = np.random.rand(n_samples, 2)
X_categorical   = np.random.choice(['Paris', 'Marseille', 'Belfort'], size=(n_samples, 1))

df_mixte = pd.DataFrame(
    np.hstack((X_numeric, X_categorical)),
    columns=['X1', 'X2', 'Ville']
)

# Correction des types : np.hstack convertit tout en str
num_cols = ['X1', 'X2']
cat_cols = ['Ville']
df_mixte[num_cols] = df_mixte[num_cols].astype(float)

print(f"Dataset mixte — shape : {df_mixte.shape}")
df_mixte.head()

Dataset mixte — shape : (100, 3)


,X1,X2,Ville
0,0.374540,0.950714,Belfort
1,0.731994,0.598658,Paris
2,0.156019,0.155995,Marseille
3,0.058084,0.866176,Marseille
4,0.601115,0.708073,Belfort


In [ ]:
# Construction de la pipeline Niveau 2
pipe_niveau2 = Pipeline(steps=[
    ("Preprocessing", ColumnTransformer(
        transformers=[
            ("Numerique",     MinMaxScaler(),                        num_cols),
            ("Categoriel",    OneHotEncoder(handle_unknown="ignore"), cat_cols)
        ],
        remainder="drop"
    ))
])

# Application
X_niveau2 = pipe_niveau2.fit_transform(df_mixte)

print(f"Shape d'entrée  : {df_mixte.shape}")
print(f"Shape de sortie : {X_niveau2.shape}")
print("(2 colonnes numériques + 3 modalités OneHot pour 'Ville' = 5 colonnes)")

pipe_niveau2

Shape d'entrée  : (100, 3)
Shape de sortie : (100, 5)
(2 colonnes numériques + 3 modalités OneHot pour 'Ville' = 5 colonnes)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('Preprocessing', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Numerique', ...), ('Categoriel', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

---

### Exercice 3 — Pipeline Niveau 3 : Prétraitement Mixte + Sélection de Variables

**Objectif :** Étendre la pipeline précédente en ajoutant une étape de sélection des `k` meilleures caractéristiques après le prétraitement mixte.

```
Données mixtes
     ├── Variables numériques  ──> MinMaxScaler ──┐
     └── Variables catégorielles ──> OneHotEncoder ─┤──> SelectKBest(k=5)
```

In [ ]:
# Génération d'un dataset mixte avec variable cible
np.random.seed(42)
n_samples = 100

X_numeric     = np.random.rand(n_samples, 2)
X_categorical = np.random.choice(['A', 'B', 'C', 'D'], size=(n_samples, 2))
y_target      = np.random.rand(n_samples) * 10

df_niv3 = pd.DataFrame(
    np.hstack((X_numeric, X_categorical)),
    columns=['X1', 'X2', 'Cat1', 'Cat2']
)
df_niv3['y'] = y_target

num_cols_3 = ['X1', 'X2']
cat_cols_3 = ['Cat1', 'Cat2']
df_niv3[num_cols_3] = df_niv3[num_cols_3].astype(float)

X_niv3 = df_niv3.drop(columns=['y'])
y_niv3 = df_niv3['y']

print(f"Dataset Niveau 3 — X : {X_niv3.shape}, y : {y_niv3.shape}")
X_niv3.head()

Dataset Niveau 3 — X : (100, 4), y : (100,)


,X1,X2,Cat1,Cat2
0,0.374540,0.950714,D,D
1,0.731994,0.598658,C,A
2,0.156019,0.155995,D,D
3,0.058084,0.866176,B,D
4,0.601115,0.708073,D,B


In [ ]:
# Construction de la pipeline Niveau 3
pipe_niveau3 = Pipeline(steps=[
    ("Preprocessing", ColumnTransformer(
        transformers=[
            ("Numerique",   MinMaxScaler(),                        num_cols_3),
            ("Categoriel",  OneHotEncoder(handle_unknown="ignore"), cat_cols_3)
        ],
        remainder="drop"
    )),
    ("SelectionFeatures", SelectKBest(score_func=f_regression, k=5))
])

# Application
X_out_niv3 = pipe_niveau3.fit_transform(X_niv3, y_niv3)

print(f"Shape avant sélection : {pipe_niveau3.named_steps['Preprocessing'].transform(X_niv3).shape}")
print(f"Shape après sélection : {X_out_niv3.shape}")

pd.DataFrame(X_out_niv3).head()

Shape avant sélection : (100, 10)
Shape après sélection : (100, 5)


,0,1,2,3,4
0,0.0,0.0,1.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,1.0,0.0,0.0
3,0.0,1.0,0.0,0.0,0.0
4,0.0,0.0,1.0,1.0,0.0


---

### Exercice 4 — Pipeline Niveau 4 : Pipeline de Bout en Bout (End-to-End)

**Objectif :** Intégrer un modèle de régression linéaire directement dans la pipeline, produisant une chaîne complète de l'entrée brute jusqu'à la prédiction.

```
Données mixtes brutes
     ├── Numérique  ──> MinMaxScaler ──┐
     └── Catégoriel ──> OneHotEncoder ─┤──> SelectKBest(k=5) ──> LinearRegression ──> ŷ
```

In [ ]:
# Construction de la pipeline Niveau 4 (bout en bout)
pipe_niveau4 = Pipeline(steps=[
    ("Preprocessing", ColumnTransformer(
        transformers=[
            ("Numerique",   MinMaxScaler(),                        num_cols_3),
            ("Categoriel",  OneHotEncoder(handle_unknown="ignore"), cat_cols_3)
        ],
        remainder="drop"
    )),
    ("SelectionFeatures", SelectKBest(score_func=f_regression, k=5)),
    ("Modele",            LinearRegression())
])

# Entraînement de la pipeline complète
pipe_niveau4.fit(X_niv3, y_niv3)

# Prédiction
y_pred = pipe_niveau4.predict(X_niv3)

# Métriques d'évaluation
r2  = r2_score(y_niv3, y_pred)
mse = mean_squared_error(y_niv3, y_pred)

print("=" * 40)
print("       Évaluation du modèle (train set)")
print("=" * 40)
print(f"  Coefficient de détermination R² : {r2:.4f}")
print(f"  Erreur quadratique moyenne MSE  : {mse:.4f}")
print("=" * 40)
print("\nNote : ces métriques sont calculées sur le jeu d'entraînement.")
print("Pour une évaluation rigoureuse, utiliser un jeu de test ou une validation croisée.")

# Aperçu des prédictions
resultats = pd.DataFrame({
    "y_réel"  : y_niv3.values[:10],
    "y_prédit": y_pred[:10]
}).round(4)
print(f"\nAperçu des 10 premières prédictions :")
resultats

       Évaluation du modèle (train set)
  Coefficient de détermination R² : 0.0893
  Erreur quadratique moyenne MSE  : 7.7636

Note : ces métriques sont calculées sur le jeu d'entraînement.
Pour une évaluation rigoureuse, utiliser un jeu de test ou une validation croisée.

Aperçu des 10 premières prédictions :


,y_réel,y_prédit
0,0.5168,3.9968
1,5.3135,5.0353
2,5.4064,3.9968
3,6.3743,6.3061
4,7.2609,4.2118
5,9.7585,5.0353
6,5.1630,6.3061
7,3.2296,5.0353
8,7.9519,5.4211
9,2.7083,4.2118


In [ ]:
# Affichage de la pipeline complète
pipe_niveau4

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('Preprocessing', ...), ('SelectionFeatures', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Numerique', ...), ('Categoriel', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output o

---

## Synthèse et Points Clés

| Concept | Points essentiels |
|---|---|
| **Partitionnement** | Toujours séparer train/test **avant** toute transformation |
| **Encodage ordinal** | Utiliser `OrdinalEncoder` avec `categories=` pour garantir le bon ordre |
| **Data leakage** | `.fit()` uniquement sur le train set ; `.transform()` uniquement sur le test set |
| **Pipeline** | Enchaîne les transformations ; gère automatiquement fit/transform |
| **ColumnTransformer** | Applique des transformations différentes selon les colonnes |
| **Pipeline end-to-end** | Intègre le modèle ; `pipeline.predict()` applique tout automatiquement |
